# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library. The dataset covers ordered logistic regression results, socio-demographics, and adoption predictors for rangeland management in Northern Kenya.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Instantiate the dataset from the URL
dataset = mlc.Dataset(croissant_url)

# Show dataset metadata (as object, not subscripted)
meta = dataset.metadata
print(f"Dataset name: {meta.name}\nDescription: {meta.description}\nIdentifier: {meta.identifier}")
print(f"Publication Date: {meta.datePublished}\nLicense: {meta.license}\nVersion: {meta.version}")

## 2. Data Overview
Review available record sets, their fields, columns, and respective `@id`s.

We'll enumerate all record sets and fields, referencing them by their Croissant `@id` attributes.

In [ ]:
# List all record sets and display their metadata
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets discovered in this dataset metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs.metadata['@id']} - {rs.metadata.get('name', '(no name)')}")
        for field in rs.fields:
            print(f"  Field: {field.metadata['@id']} - {field.metadata.get('name', '(no name)')}")
        for col in rs.columns:
            print(f"  Column: {col.metadata['@id']} - {col.metadata.get('name', '(no name)')}")

## 3. Data Extraction
Load the available record sets into DataFrames for analysis. We'll use the `@id` for each record set and field.

For demonstration, we'll extract the first available record set (if any).

In [ ]:
# Gather the list of record set @ids
record_set_ids = [rs.metadata['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# List what's loaded
if dataframes:
    print(f"Loaded DataFrames: {list(dataframes.keys())}")
    main_rs_id = record_set_ids[0]
    print(f"Columns for record set '@id': {main_rs_id}")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No record sets to load as DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing steps such as filtering, normalizing, and grouping. All columns and fields referenced by `@id`.

If the dataset has a numeric field, we will demonstrate filtering and normalization.

In [ ]:
# Pick a numeric field to work on from the main record set
if dataframes:
    main_df = dataframes[main_rs_id]
    numeric_fields = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Selected numeric field: {numeric_field}")
        threshold = main_df[numeric_field].mean() if main_df[numeric_field].dropna().size else 0
        filtered_df = main_df[main_df[numeric_field] > threshold].copy()

        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the field
        if filtered_df[numeric_field].std() != 0:
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        else:
            print(f"Standard deviation of {numeric_field} is zero; cannot normalize.")

        # Pick a group field if available (categorical)
        non_numeric_fields = [col for col in main_df.columns if not pd.api.types.is_numeric_dtype(main_df[col])]
        group_field = non_numeric_fields[0] if non_numeric_fields else None
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No categorical/group field available for grouping.")
    else:
        print("No numeric fields available in data for EDA.")
else:
    print("No data to analyze.")

## 5. Visualization
Visualize data distributions or relationships between dataset fields using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the numeric field
if dataframes and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If group/categorical field is available, plot mean values
    if group_field:
        order = filtered_df[group_field].value_counts().index
        plt.figure(figsize=(10,4))
        sns.barplot(
            data=filtered_df,
            x=group_field,
            y=numeric_field,
            order=order
        )
        plt.xticks(rotation=45)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
This notebook illustrated how to load and explore a FAIR dataset using the `mlcroissant` library with all references by `@id`. Key exploration steps included examining record sets, extracting records, filtering and normalizing numeric variables, and visualizing data distributions. Further analysis can be conducted by joining record sets, applying domain logic, or integrating with other data science workflows.